# Mathematical foundations

This chapter is a self-contained review of the mathematics that the rest of
Part II builds on. If you are comfortable with linear projections, gradient
descent, the exponential family, and generalized linear models, you can skip to
[](functional_gradient_boosting.ipynb). Otherwise, read this first. Every later
chapter assumes the vocabulary and results collected here.

```{admonition} How to read this chapter
:class: tip
Each section states the **idea** in one paragraph, develops it with the **key
formula(s)**, and closes with **why it matters for boosting**. Work the short
exercises; they are chosen to expose the specific subtleties that recur in the
boosting literature.
```



## 1. Linear algebra: projections and hat matrices



### 1.1 The ordinary least-squares fit as a projection

Given a design matrix $X \in \mathbb{R}^{n \times p}$ (rows = observations,
columns = predictors) and a response $y \in \mathbb{R}^n$, the ordinary
least-squares (OLS) coefficients solve

$$
\hat\beta = \argmin_\beta \|y - X\beta\|^2 = (X^\top X)^{-1} X^\top y,
$$

assuming $X$ has full column rank so that $X^\top X$ is invertible. The fitted
values are

$$
\hat y = X\hat\beta = \underbrace{X(X^\top X)^{-1} X^\top}_{\displaystyle H}\, y.
$$

The matrix $H$ is the **hat matrix** (it "puts the hat on $y$"). It is an
*orthogonal projector*: $H$ maps $y$ onto the column space $\mathcal{C}(X)$, the
closest point in that subspace.


```{figure} ../figures/mermaid-e7ad66e0ea65.svg
:name: mermaid-e7ad66e0ea65
:alt: Projection of the response vector into fitted values in the column space of the design matrix and a residual orthogonal to that space.

The hat matrix decomposes the response into its projection onto the design space and an orthogonal residual.
```

Three properties of $H$ are used constantly in boosting:

1. **Symmetry and idempotence.** $H^\top = H$ and $H^2 = H$. Applying the
   projection twice is the same as applying it once.
2. **Trace equals rank.** $\operatorname{tr}(H) = \operatorname{rank}(X)$. The
   trace counts the number of "directions" the projection keeps — this is the
   **degrees of freedom** of an unpenalized fit.
3. **Residual orthogonality.** $(I - H)y \perp \mathcal{C}(X)$, i.e.
   $X^\top(I - H)y = 0$. The residuals are orthogonal to every column of the
   design.

````{admonition} Exercise 1.1
:class: tip
Show that for the hat matrix $H = X(X^\top X)^{-1} X^\top$, we have
$H^\top = H$ and $H^2 = H$. *Hint: expand $H^2$ and cancel.*

```{admonition} Answer
:class: dropdown
$H^\top = \big[(X^\top X)^{-1}\big]^\top X^\top = X(X^\top X)^{-1} X^\top = H$,
since $X^\top X$ is symmetric. For idempotence:
$H^2 = X\underbrace{(X^\top X)^{-1} X^\top X}_{=\,I}(X^\top X)^{-1} X^\top = H$.
```
````



### 1.2 Weighted least squares

Boosting fits base learners to **weighted** data (the weights come from the
loss family and, in cross-validation, from the resampling scheme). With weights
$w_i \ge 0$ collected in $W = \operatorname{diag}(w)$, the weighted OLS
coefficients are

$$
\hat\beta = (X^\top W X)^{-1} X^\top W y,
$$

and the weighted hat matrix is

$$
H_W = X(X^\top W X)^{-1} X^\top W.
$$

Note $H_W$ is **not symmetric** in the standard inner product when $W \ne I$,
but it *is* symmetric in the $W$-inner product:
$\langle a, b\rangle_W = a^\top W b$. It is still idempotent ($H_W^2 = H_W$),
and its trace $\operatorname{tr}(H_W) = \operatorname{rank}(X)$ still equals the
number of free columns. Every boosting iteration fits the negative gradient
by weighted least squares, regardless of the loss function.



### 1.3 Ridge regression and penalized least squares

When $p$ is large relative to $n$, or when we want to shrink coefficients,
OLS overfits. **Ridge regression** adds a penalty on the coefficient size:

$$
\hat\beta_\lambda = \argmin_\beta \big\{ \|y - X\beta\|^2 + \lambda\, \beta^\top K \beta \big\}
                   = (X^\top X + \lambda K)^{-1} X^\top y,
$$

where $K \succeq 0$ is a positive-semidefinite **penalty matrix** and
$\lambda \ge 0$ is the smoothing parameter. The penalty term $\lambda\,\beta^\top K \beta$
discourages coefficients that are "large" in the $K$-geometry. Two special
cases:

- **$K = I$ (ridge):** shrinks every coefficient toward zero equally.
- **$K = \Delta^\top \Delta$ (difference penalty):** penalizes the $k$-th
  differences of $\beta$, shrinking toward a polynomial of degree $k-1$. This is
  the P-spline penalty used by `bbs` (see [](base_learners.ipynb)).

The penalized hat matrix is

$$
H_\lambda = X(X^\top X + \lambda K)^{-1} X^\top.
$$

Its trace $\operatorname{tr}(H_\lambda)$ decreases smoothly from
$\operatorname{rank}(X)$ (at $\lambda = 0$) toward the dimension of the null
space $\ker(K)$ (as $\lambda \to \infty$). This trace is the **effective degrees
of freedom** of the penalized fit — it measures how many "free parameters" the
smooth has, on a continuous scale. The entire [](penalties_df.ipynb) chapter is
about inverting this relationship: given a target `df`, find the $\lambda$ that
achieves it.

````{admonition} Exercise 1.2
:class: tip
Let $K = I$ (ridge). Show that as $\lambda \to \infty$, $\hat\beta_\lambda \to 0$.
What is $\operatorname{tr}(H_\lambda)$ in this limit?

```{admonition} Answer
:class: dropdown
$\hat\beta_\lambda = (X^\top X + \lambda I)^{-1} X^\top y$. As $\lambda \to \infty$,
$(X^\top X + \lambda I)^{-1} \to (1/\lambda) I$, so
$\hat\beta_\lambda \to (1/\lambda) X^\top y \to 0$. The hat matrix
$H_\lambda = X(X^\top X + \lambda I)^{-1} X^\top \to (1/\lambda) X X^\top$,
whose trace $(1/\lambda)\operatorname{tr}(XX^\top) = (1/\lambda)\|X\|_F^2 \to 0$.
For ridge, the null space of $K = I$ is $\{0\}$, so the df can reach zero.
```
````



### 1.4 Eigendecomposition and the spectral view

Penalized regression is best understood through **eigenvalues**. Consider the
generalized eigenvalue problem

$$
K v_\ell = d_\ell\, (X^\top X)\, v_\ell, \qquad \ell = 1, \dots, p,
$$

where $d_\ell \ge 0$ are the generalized eigenvalues and $v_\ell$ are the
eigenvectors, orthonormalized so that $v_\ell^\top X^\top X v_m = \delta_{\ell m}$.
In this basis, the penalized hat matrix decomposes as

$$
H_\lambda = \sum_{\ell=1}^{p} \underbrace{\frac{1}{1 + \lambda\, d_\ell}}_{\displaystyle h_\ell(\lambda)}\, P_\ell,
$$

where $P_\ell$ are rank-one projectors onto the eigendirections. The **shrinkage
factor** $h_\ell(\lambda) = 1/(1 + \lambda d_\ell) \in (0, 1]$ tells us how
much of direction $\ell$ survives the penalty:

- **$d_\ell = 0$ (null space of $K$):** $h_\ell = 1$ for all $\lambda$. The
  direction is **unpenalized** — it survives fully. (For a second-order
  difference penalty, the null space is the linear functions.)
- **$d_\ell$ large:** $h_\ell \approx 1/(\lambda d_\ell)$ for moderate $\lambda$.
  The direction is **heavily shrunk**. (High-frequency wiggles in a P-spline
  have large $d_\ell$ and are smoothed away.)

The shrinkage factor $h_\ell(\lambda) = 1/(1 + \lambda d_\ell)$ as a function
of the eigenvalue index $\ell$. Null-space directions ($d_\ell = 0$) are never
shrunk ($h_\ell = 1$); high-frequency directions ($d_\ell$ large) are driven to
zero:

```{figure} ../figures/fig-shrinkage-spectrum.svg
:label: fig-shrinkage
:alt: Shrinkage factor vs eigenvalue index

How a penalty acts on each eigendirection. Null-space directions have $d_\ell = 0$ and pass through untouched; the higher the eigenvalue, the harder that direction is shrunk toward zero.
```

Increasing $\lambda$ shrinks directions with large $d_\ell$ more strongly,
reducing rapid variation in the fitted P-spline while retaining the unpenalized
polynomial directions. The degrees-of-freedom formula
$\mathrm{df}(\lambda) = \sum_\ell (2 h_\ell - h_\ell^2)$ sums the
contributions of these directions; [](penalties_df.ipynb) gives the derivation.



## 2. Optimization: gradient descent in vector space



### 2.1 The descent principle

To minimize a differentiable function $F: \mathbb{R}^d \to \mathbb{R}$, the
**steepest-descent** direction at a point $x$ is $-\nabla F(x)$, because the
directional derivative

$$
\frac{d}{dt} F(x + t\, d)\bigg|_{t=0} = \nabla F(x)^\top d
$$

is most negative when $d = -\nabla F(x)/\|\nabla F(x)\|$. **Gradient descent**
takes small steps along this direction:

$$
x^{[m]} = x^{[m-1]} - \nu\, \nabla F(x^{[m-1]}),
$$

where $\nu > 0$ is the step length. For a convex $F$ with Lipschitz-continuous
gradient, $F(x^{[m]})$ decreases monotonically and converges to the global
minimum, with rate $O(1/m)$ for fixed small $\nu$.

Each step moves along $-\nabla F$, perpendicular to the contours of $F$,
reducing the function value monotonically toward the minimum $x^*$:

```{figure} ../figures/fig-gradient-descent.svg
:label: fig-gd
:alt: Gradient descent: F(x) decreases monotonically each step

Gradient descent on a convex objective. Each step moves along the negative gradient, perpendicular to the contours, and the objective decreases monotonically toward the minimum.
```

```{admonition} Why this matters for boosting
:class: important
Boosting **is** gradient descent — but in *function space*, not parameter space.
The function being minimized is the empirical risk
$\mathcal{R}(f) = \sum_i w_i \rho(y_i, f(x_i))$, and instead of taking a step
along $-\nabla \mathcal{R}$ directly (which would give an unconstrained
function), boosting projects the gradient onto the span of a **base learner**
and takes a small step along that. Section 3 of
[](functional_gradient_boosting.ipynb) makes this precise.
```



### 2.2 Coordinate descent and component-wise updates

**Coordinate descent** minimizes $F$ by updating one coordinate at a time,
holding the others fixed. For convex $F$ it converges to the global minimum; for
nonconvex $F$ it converges to a stationary point. Component-wise boosting is a
*greedy coordinate descent*: at each step, it picks the coordinate (base learner)
that gives the largest risk reduction, rather than cycling through them in a
fixed order.

```{figure} ../figures/mermaid-2d1dde2fcdb5.svg
:name: mermaid-2d1dde2fcdb5
:alt: Path from an initial coefficient vector to the optimum through successive updates of the best coordinate.

Greedy coordinate descent repeatedly evaluates candidate coordinates and updates the direction that most reduces the objective.
```

The advantage of the greedy choice is **automatic variable selection**: a
coordinate (base learner) that never reduces the risk is never selected, so its
coefficient stays exactly zero. This is why boosting produces sparse, interpretable
models even when $p \gg n$.



### 2.3 Convexity and the role of the step length $\nu$

A function $F$ is **convex** if

$$
F(t x + (1-t) x') \le t F(x) + (1-t) F(x') \qquad \forall\, t \in [0,1].
$$

For convex $F$, gradient descent with sufficiently small $\nu$ converges to the
global minimum from any starting point. For **strongly convex** $F$ (i.e.,
$F - \frac{\mu}{2}\|x\|^2$ is convex for some $\mu > 0$), convergence is linear:
$F(x^{[m]}) - F^* \le (1 - \nu\mu)^m (F(x^{[0]}) - F^*)$.

Gaussian, Binomial and Poisson losses are convex and differentiable in their
predictors. Quantile loss is convex but nonsmooth, so its updates use a
subgradient. For a differentiable loss and a fitted learner that gives a
descent direction, a sufficiently small step decreases risk locally.
The default $\nu=0.1$ is not a universal monotonicity guarantee for nonsmooth
or nonconvex losses; inspect the recorded risk path. The conditions for smooth
losses are given in [](theory_properties.ipynb).



## 3. Probability and statistics: the exponential family



### 3.1 The exponential family

A distribution belongs to the **exponential family** if its density can be
written

$$
d(y \mid \theta) = \exp\!\big\{\,\theta\, T(y) - A(\theta) + B(y)\,\big\},
$$

where:
- $\theta$ is the **natural (canonical) parameter**,
- $T(y)$ is the **sufficient statistic**,
- $A(\theta) = \log \int \exp\{\theta T(y) + B(y)\}\, dy$ is the
  **log-partition function** (ensures the density integrates to 1),
- $B(y)$ is a base measure.

The exponential family includes the Gaussian, Bernoulli, Poisson, Gamma,
Binomial, and many others. Three properties make it central to boosting:

1. **Mean and variance from $A$.** $\mathbb{E}[T(Y)] = A'(\theta)$ and
   $\operatorname{Var}[T(Y)] = A''(\theta)$. The variance function is the
   *derivative of the mean* with respect to $\theta$.
2. **The score is simple.** The derivative of the log-likelihood with respect to
   $\theta$ is $\partial \log d / \partial \theta = T(y) - A'(\theta) =
   T(y) - \mathbb{E}[T(Y)]$. The score is the *centered sufficient statistic*.
3. **Convexity.** $A(\theta)$ is convex (its second derivative is the variance,
   which is non-negative), so maximum likelihood is a convex problem for
   exponential-family models.

````{admonition} Example: Gaussian as exponential family
:class: tip

The $N(\mu, \sigma^2)$ density with **known** $\sigma^2$ is

$$
d(y \mid \mu) = \frac{1}{\sqrt{2\pi\sigma^2}}
   \exp\!\Big\{-\frac{(y-\mu)^2}{2\sigma^2}\Big\}
 = \exp\!\Big\{\underbrace{\frac{\mu}{\sigma^2}}_{\theta}\, y
   - \underbrace{\frac{\mu^2}{2\sigma^2}}_{A(\theta)}
   + \underbrace{\Big(-\frac{y^2}{2\sigma^2} - \tfrac{1}{2}\log(2\pi\sigma^2)\Big)}_{B(y)}\Big\}.
$$

So $\theta = \mu/\sigma^2$, $T(y) = y$, $A(\theta) = \sigma^2\theta^2/2$, and
$A'(\theta) = \sigma^2\theta = \mu$. The score is
$\partial\log d/\partial\theta = y - \mu$, which is exactly the **residual** —
the negative gradient of the Gaussian loss used in L2Boosting.
````



### 3.2 Maximum likelihood and Fisher information

Given iid data $y_1, \dots, y_n$ from $d(y \mid \theta)$, the **maximum
likelihood estimator** (MLE) maximizes the log-likelihood

$$
\hat\theta = \argmax_\theta \sum_{i=1}^n \log d(y_i \mid \theta).
$$

The first-order condition sets the **score** to zero:
$\sum_i \partial_\theta \log d(y_i \mid \hat\theta) = 0$. The **Fisher
information** measures the curvature of the log-likelihood:

$$
\mathcal{I}(\theta) = -\mathbb{E}\!\Big[\frac{\partial^2}{\partial\theta^2} \log d(Y \mid \theta)\Big]
                     = \operatorname{Var}\!\Big[\frac{\partial}{\partial\theta}\log d(Y \mid \theta)\Big].
$$

The MLE is asymptotically normal: $\sqrt{n}(\hat\theta - \theta) \to N(0, \mathcal{I}^{-1})$.
The Fisher information determines standard errors and confidence intervals
and appears in the Cramér–Rao bound.

```{admonition} Why this matters for boosting
:class: important
Boosting minimizes the **negative log-likelihood** (the empirical risk) by
gradient descent. The "negative gradient" it fits at each step is exactly the
**score function**. For exponential-family losses, the score has a clean form
(residual for Gaussian, $y - p$ for Bernoulli, $y - \mu$ for Poisson), which is
why the boosting pseudo-residuals look familiar. The Fisher information appears
in the GAMLSS boosting update as the natural *weight* for each parameter — see
the Fisher-scoring interpretation in [](gamlss.ipynb).
```



## 4. Generalized linear models (GLMs)



### 4.1 The GLM setup

A **generalized linear model** (GLM) {cite}`mccullagh1989generalized` extends
linear regression to non-Gaussian responses. It has three components:

1. **The linear predictor:** $\eta_i = x_i^\top \beta$ — a linear combination of
   covariates.
2. **The link function:** $g(\mu_i) = \eta_i$, where $\mu_i = \mathbb{E}[Y_i \mid x_i]$
   and $g$ is monotonic and differentiable. The link connects the mean of the
   response to the linear predictor.
3. **The exponential-family distribution:** $Y_i \mid x_i \sim \text{ExpFam}(\mu_i, \phi)$,
   where $\phi$ is a dispersion parameter.

```{figure} ../figures/mermaid-1415e9049cb4.svg
:name: mermaid-1415e9049cb4
:alt: Generalized linear model pipeline from covariates through the linear predictor and link to the conditional mean and response distribution.

A generalized linear model maps covariates to a linear predictor, transforms it to a conditional mean through the inverse link, and specifies a response distribution.
```

Common GLMs:

| Response type | Distribution | Link $g$ | Name |
|---|---|---|---|
| Continuous, unbounded | Gaussian | identity | Linear regression |
| Binary | Bernoulli | logit | Logistic regression |
| Count | Poisson | log | Poisson regression |
| Positive, skewed | Gamma | log | Gamma regression |
| Positive, skewed | Inverse Gaussian | $1/\mu^2$ | Inverse Gaussian reg. |



### 4.2 Fitting GLMs: iteratively reweighted least squares (IRLS)

GLMs are fit by **IRLS**, which iterates weighted least squares. At iteration
$t$:

1. Compute the **working response** $z_i^{(t)} = \eta_i^{(t)} + (y_i - \mu_i^{(t)})\, g'(\mu_i^{(t)})$.
2. Compute the **working weights** $W_i^{(t)} = 1 / \big[\operatorname{Var}(Y_i)\, (g'(\mu_i^{(t)}))^2\big]$.
3. Solve the weighted least-squares problem
   $\beta^{(t+1)} = (X^\top W^{(t)} X)^{-1} X^\top W^{(t)} z^{(t)}$.

IRLS is exactly **Newton's method** on the GLM log-likelihood, with the
Hessian replaced by its expected value (the Fisher information). It converges
in a handful of iterations for well-conditioned problems.

````{admonition} Why this matters for boosting
:class: important

A single IRLS step has the same form as **one boosting step with a linear base
learner**:

$$
\beta^{(t+1)} = (X^\top W X)^{-1} X^\top W z,
$$

where $z$ is the working response and $W$ the working weights. In boosting,
$z$ is the **negative gradient** (pseudo-residual) and $W$ is determined by the
family. The difference:

- **IRLS** takes the **full Newton step** (converges in a few iterations) but
  uses the **entire** design $X$ simultaneously.
- **Boosting** takes a **small step** ($\nu \ll 1$) using **one base learner at
  a time** (component-wise), iterating many times.

Boosting is thus a **step-length-damped, component-wise, incremental** version of
IRLS / Fisher scoring. This connection is made rigorous in
[](gamlss.ipynb) (Fisher scoring interpretation) and [](theory_properties.ipynb)
(convergence). The `ngradient_and_risk` fusion in `mboostrs` computes exactly the
working response and working weights that IRLS would use.
````



## 5. From GLMs to GAMs: adding smoothness



### 5.1 The limitation of linearity

A GLM assumes the effect of each covariate is **linear** (on the link scale):
$g(\mu) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$. Real-world effects are
often nonlinear: a drug's effect may peak at a dose and decline, age may have a
U-shaped effect on mortality, time may have a seasonal pattern.

**Generalized additive models** (GAMs) {cite}`hastie1990generalized` replace
each linear term $\beta_j x_j$ with a **smooth function** $f_j(x_j)$:

$$
g(\mu_i) = \beta_0 + f_1(x_{i1}) + f_2(x_{i2}) + \dots + f_p(x_{ip}).
$$

The model is still *additive* (no interactions unless explicitly added), but each
$f_j$ can be an arbitrary smooth function — a GLM captures only linear trends,
while a GAM captures nonlinear patterns:

```{figure} ../figures/fig-glm-vs-gam.svg
:label: fig-glm-gam
:alt: GLM linear effect vs GAM smooth effect

The same data under a linear effect and a smooth one. A GLM can only tilt the line; a GAM follows the curvature while staying additive.
```



### 5.2 Representing smooth functions: basis expansions

A smooth function $f(x)$ is represented by a **basis expansion**:

$$
f(x) = \sum_{j=1}^{p} \beta_j\, B_j(x),
$$

where $B_1, \dots, B_p$ are **basis functions** and $\beta_1, \dots, \beta_p$ are
coefficients. Common bases:

- **Polynomials:** $B_j(x) = x^{j-1}$. Simple but unstable for high degree
  (Runge's phenomenon).
- **B-splines:** piecewise polynomials with compact support. Numerically
  stable, locally supported, and the default in `mboost`/`mboostrs` via the
  `bbs` learner. See [](base_learners.ipynb) for the Cox–de Boor recursion.
- **Radial basis functions:** $B_j(x) = \phi(\|x - c_j\|)$ for centers $c_j$.
  Used in kernel methods.

Once a basis is chosen, fitting a GAM reduces to a **penalized GLM**: the design
matrix $X$ has columns $B_j(x_i)$, and a penalty $K$ controls smoothness. The
fit is by penalized IRLS — the same machinery as GLMs, plus the penalty.

```{admonition} Why this matters for boosting
:class: important
A GAM is a GLM with a **rich, penalized** design matrix. Boosting takes this one
step further: instead of fitting all smooth terms simultaneously (which requires
choosing a penalty for each), it **adds one smooth term at a time**, and the
selection process determines which terms matter and how complex each should be.
Each iteration fits the candidate learners (`bbs`, `bols`, `bmono`, …) to the
current negative gradient and adds a shrunken fit from the selected learner. [](from_glm_to_boosting.ipynb) traces this evolution in detail.
```



### 5.3 The bias-variance tradeoff in smoothing

A smooth function $f(x)$ estimated from data faces a fundamental tradeoff:

- **Under-smoothing** (too much flexibility, small $\lambda$): the fit captures
  noise in the data $\rightarrow$ high variance, low bias.
- **Over-smoothing** (too little flexibility, large $\lambda$): the fit misses
  real structure $\rightarrow$ low variance, high bias.

```{figure} ../figures/fig-bias-variance.svg
:label: fig-bias-variance
:alt: Bias-variance tradeoff as a function of model flexibility

Error against model flexibility. Too little flexibility misses real structure (bias); too much fits the noise (variance); the total error is minimized between them.
```

The red dot marks the minimum total error in this illustration.

In `mboost`/`mboostrs`, two parameters control this tradeoff:
the per-learner `df` (flexibility of each base learner) and the global
$m_{\text{stop}}$ (number of boosting iterations). The standard practice is to
fix `df` small (making each learner weak) and tune $m_{\text{stop}}$ by
cross-validation — this is the regularization strategy that makes boosting
work, as developed in [](functional_gradient_boosting.ipynb) and
[](model_selection.ipynb).



## 6. Summary: the conceptual chain

The rest of Part II builds on this chain of ideas:

```{figure} ../figures/mermaid-b29e70643adb.svg
:name: mermaid-b29e70643adb
:alt: Conceptual progression from ordinary least squares and gradient descent through penalized smooth base learners and component-wise boosting to GAMLSS.

Component-wise boosting connects projection and penalized smoothing with gradient descent in function space; distributional boosting extends the construction to multiple parameters.
```

Each arrow is a generalization that retains the core idea: **project the
gradient of the loss onto a structured subspace and take a small step**. The
subspace grows from "one coordinate" (coordinate descent) to "one smooth term"
(boosting) to "one parameter of a distribution" (GAMLSS boosting), but the
engine is the same.

The next chapter, [](from_glm_to_boosting.ipynb), traces the historical and
pedagogical path from GLMs to model-based boosting, motivating each design
choice. After that, [](functional_gradient_boosting.ipynb) gives the formal
algorithm and [](theory_properties.ipynb) proves its convergence properties.

```{bibliography}
:filter: docname in docnames
```